# 🖌️ Flood Mask Annotator (Google Drive) — no more re-uploading

Reads your tiles from **Google Drive** and writes each mask **right next to its image**
as `<name>_mask.png`. Because everything lives in Drive:

- **You upload images ONCE.** Restarting the runtime never loses them.
- **Masks save as you paint**, straight to Drive — a crash/restart just resumes where
  you left off (already-painted tiles reappear painted, ready to fix).
- **`evaluate.py` finds the masks automatically** (its `<stem>_mask.png` sidecar rule),
  so Accuracy / F1 / IoU populate with zero extra steps.

### One-time setup on drive.google.com
1. Create a folder: **`sri_lanka_floods/satellite_images_data`**
2. Put your **tile PNGs** + their **`metadata.csv`** in it (drag-drop, or drop a `.zip`
   there — cell 2 unzips it for you).
3. Run the cells below. Paint flood in red → **Save & Next**.


In [ ]:
# 1️⃣  Install ONCE  (pinned so google.colab & pandas stay intact — only unused HF pkgs warn)
!pip -q install "gradio==4.44.1" "huggingface_hub==0.25.2" "opencv-python-headless"
print("✅ installed.")
print("ℹ️ If Colab shows a 'RESTART RUNTIME' banner, click it, then run cell 2 onward.")
print("   Your Drive files are safe — you never re-upload.")

In [ ]:
# 2️⃣  Mount Google Drive & find your uploaded zips  (NO re-uploading, ever)
from google.colab import drive
drive.mount("/content/drive")

import glob, zipfile
from pathlib import Path

# ── Where are your files? ─────────────────────────────────────────────────────
# Leave DRIVE_FOLDER = None to AUTO-FIND the folder containing your zips
# (searches MyDrive 3 levels deep). Or set it explicitly, e.g.:
#   DRIVE_FOLDER = "/content/drive/MyDrive/my_upload_folder"
DRIVE_FOLDER = None

root = Path("/content/drive/MyDrive")
if DRIVE_FOLDER is None:
    hits = [p for pat in ("*.zip", "*/*.zip", "*/*/*.zip")
            for p in root.glob(pat) if "flood" in p.name.lower() or "sri_lanka" in p.name.lower()]
    assert hits, "No flood/sri_lanka zip found in MyDrive — set DRIVE_FOLDER manually."
    IMAGES_DIR = hits[0].parent
else:
    IMAGES_DIR = Path(DRIVE_FOLDER)
print("📂 working folder:", IMAGES_DIR)

# ── Extract BOTH zips (idempotent — skips files that already exist) ───────────
# images zip → tile PNGs + metadata.csv ; labels zip → <tile>_mask.png sidecars
for z in sorted(IMAGES_DIR.glob("*.zip")):
    with zipfile.ZipFile(z) as zf:
        n_new = 0
        for n in zf.namelist():
            base = Path(n).name
            if not base or n.endswith("/"):
                continue
            if base.lower().endswith(".png") or base == "metadata.csv":
                dest = IMAGES_DIR / base          # masks land beside tiles = sidecars
                if not dest.exists():
                    dest.write_bytes(zf.read(n)); n_new += 1
    print(f"  {z.name}: extracted {n_new} new file(s)")

IMAGES = sorted(p for p in glob.glob(str(IMAGES_DIR / "*.png")) if not p.endswith("_mask.png"))
done = sum((IMAGES_DIR / (Path(p).stem + "_mask.png")).exists() for p in IMAGES)
print(f"✅ {len(IMAGES)} tiles | {done} already have masks (they'll open pre-painted for fixing)")
assert IMAGES, "No tiles found — check the folder above contains your images zip."

In [ ]:
# 3️⃣  Launch the paint tool  ──────────────────────────────────────────────────
import gradio as gr, numpy as np, cv2
from pathlib import Path

def mpath(i):    return IMAGES_DIR / (Path(IMAGES[i]).stem + "_mask.png")
def read_rgb(i): return cv2.cvtColor(cv2.imread(IMAGES[i]), cv2.COLOR_BGR2RGB)

def editor_value(i):
    """Tile as background; if a mask already exists, show it in red so you just fix it."""
    img = read_rgb(i); layers = []
    p = mpath(i)
    if p.exists():
        m = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if m is not None and m.shape[:2] == img.shape[:2]:
            rgba = np.zeros((*m.shape, 4), np.uint8)
            rgba[m > 127] = (255, 0, 0, 255)
            layers = [rgba]
    return {"background": img, "layers": layers, "composite": None}

def status(i):
    done = sum(mpath(k).exists() for k in range(len(IMAGES)))
    return f"### Tile {i+1} / {len(IMAGES)}  &nbsp;|&nbsp; saved: **{done}**  &nbsp;|&nbsp; `{Path(IMAGES[i]).name}`"

def extract_mask(ev, i):
    img = read_rgb(i); h, w = img.shape[:2]
    m = np.zeros((h, w), np.uint8)
    if isinstance(ev, dict):
        for layer in (ev.get("layers") or []):
            if layer is None: continue
            a = layer[..., 3] if (layer.ndim == 3 and layer.shape[-1] == 4) else layer.reshape(h, w, -1).sum(-1)
            m[a > 10] = 255
        if m.max() == 0 and ev.get("composite") is not None:
            diff = np.abs(ev["composite"][..., :3].astype(int) - img.astype(int)).sum(-1)
            m[diff > 30] = 255
    return m

def _save(ev, i):                       # writes the sidecar mask straight to Drive
    cv2.imwrite(str(mpath(i)), extract_mask(ev, i))

def step(ev, i, d, do_save=True):
    if do_save: _save(ev, i)
    j = max(0, min(len(IMAGES) - 1, i + d))
    return editor_value(j), j, status(j)

def empty_next(ev, i):
    cv2.imwrite(str(mpath(i)), np.zeros(read_rgb(i).shape[:2], np.uint8))
    j = max(0, min(len(IMAGES) - 1, i + 1))
    return editor_value(j), j, status(j)

with gr.Blocks(title="Flood Mask Annotator") as demo:
    gr.Markdown("## 🌊 Paint FLOOD / WATER in **red**, then **Save & Next**. "
                "Eraser removes mistakes; brush size is in the toolbar. Every Save writes to Drive.")
    i_state = gr.State(0)
    info = gr.Markdown(status(0))
    editor = gr.ImageEditor(
        value=editor_value(0), type="numpy", height=650,
        brush=gr.Brush(colors=["#ff0000"], default_size=14, color_mode="fixed"),
        label="Flood annotation")
    with gr.Row():
        prev_b  = gr.Button("⟵ Save & Prev")
        empty_b = gr.Button("🚫 No flood here → empty mask & Next")
        next_b  = gr.Button("Save & Next ⟶", variant="primary")
    next_b.click(lambda ev, i: step(ev, i, +1), [editor, i_state], [editor, i_state, info])
    prev_b.click(lambda ev, i: step(ev, i, -1), [editor, i_state], [editor, i_state, info])
    empty_b.click(empty_next, [editor, i_state], [editor, i_state, info])

demo.launch(share=True, debug=False)

In [ ]:
# 4️⃣  (optional) progress + red-overlay previews in a _preview/ subfolder
import glob, cv2, numpy as np
from pathlib import Path
PREV = IMAGES_DIR / "_preview"; PREV.mkdir(exist_ok=True)
done = 0
for p in IMAGES:
    mp = IMAGES_DIR / (Path(p).stem + "_mask.png")
    if not mp.exists(): continue
    done += 1
    img = cv2.imread(p); m = cv2.imread(str(mp), 0)
    if m is None: continue
    sel = m > 127
    img[sel] = (0.5 * img[sel] + np.array([0, 0, 127])).clip(0, 255).astype(np.uint8)  # BGR red
    cv2.imwrite(str(PREV / ("preview_" + Path(p).stem + ".png")), img)
print(f"{done}/{len(IMAGES)} tiles annotated. Overlays saved in {PREV}")

## ✅ Now run `evaluate.py` against these masks

Your masks are sidecars next to the images in Drive — exactly what the evaluator
auto-discovers. Point it at that same folder:

```bash
python evaluate.py --load-in-4bit \
  --images-dir "/content/drive/MyDrive/sri_lanka_floods/satellite_images_data" \
  --metadata   "/content/drive/MyDrive/sri_lanka_floods/satellite_images_data/metadata.csv"
```

**Accuracy / F1 / IoU / Dice will populate automatically** (no more `N/A`), because
every `<tile>_mask.png` now sits beside its `<tile>.png`.

> Tip: you don't need all tiles labelled. Do **20–50 flood tiles** first, run the
> evaluator with `--max-images`, and you'll already get real metrics.
